In [1]:
from xaikd import models, utils, constants

import torch

/home/pat/projects/xai-kd/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
utils.count_params_in_model

<function xaikd.utils.count_params_in_model(model: torch.nn.modules.module.Module) -> Tuple[int, int]>

In [3]:
model_l = models.get_trained_model("imagenet-mobilenetl-tv")

In [4]:
model_l.features[:2]

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
    (2): Hardswish()
  )
  (1): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      )
    )
  )
)

In [5]:
model_s = models.get_trained_model("imagenet-mobilenets-tv")
model_s.features[2:]

Sequential(
  (2): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(16, 72, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(72, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (1): Conv2dNormActivation(
        (0): Conv2d(72, 72, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=72, bias=False)
        (1): BatchNorm2d(72, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (2): Conv2dNormActivation(
        (0): Conv2d(72, 24, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(24, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      )
    )
  )
  (3): InvertedResidual(
    (block): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(24, 88, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(88, eps=0.001, momentum=

In [6]:
def pretty_print(model):    
    num_features = len(model.features)
    print(f"----- ({num_features} features) ------")
    
    x = torch.randn(1, 3, 224, 224)
    _, arr_outputs = utils.interceptor.forward_and_intercept_intermediate_layers(
        model, x,
        layers=[f"features.{ix}" for ix in range(num_features)],
        detach_output=False
    )

    for lix, output in enumerate(arr_outputs):
        print(f"features.{lix}: {output.shape}")

pretty_print(models.get_trained_model("imagenet-mobilenetl-tv"))

----- (17 features) ------
features.0: torch.Size([1, 16, 112, 112])
features.1: torch.Size([1, 16, 112, 112])
features.2: torch.Size([1, 24, 56, 56])
features.3: torch.Size([1, 24, 56, 56])
features.4: torch.Size([1, 40, 28, 28])
features.5: torch.Size([1, 40, 28, 28])
features.6: torch.Size([1, 40, 28, 28])
features.7: torch.Size([1, 80, 14, 14])
features.8: torch.Size([1, 80, 14, 14])
features.9: torch.Size([1, 80, 14, 14])
features.10: torch.Size([1, 80, 14, 14])
features.11: torch.Size([1, 112, 14, 14])
features.12: torch.Size([1, 112, 14, 14])
features.13: torch.Size([1, 160, 7, 7])
features.14: torch.Size([1, 160, 7, 7])
features.15: torch.Size([1, 160, 7, 7])
features.16: torch.Size([1, 960, 7, 7])


In [7]:
def pretty_print_student(student_name):
    
    model = models.get_untrained_model(student_name, num_classes=7)
    nparams, n_trainable_params = utils.count_params_in_model(model)
   
    num_features = len(model.features)
    print(f"Student `{student_name}` (params: {nparams / 1e6:.1f}M)")
    
    x = torch.randn(1, 3, 224, 224)
    _, arr_outputs = utils.interceptor.forward_and_intercept_intermediate_layers(
        model, x,
        layers=[f"features.{ix}" for ix in range(num_features)],
        detach_output=False
    )

    for lix, output in enumerate(arr_outputs):
        print(f"> features.{lix}: {output.shape}")

pretty_print_student("student-mobilenets")

Student `student-mobilenets` (params: 1.5M)
> features.0: torch.Size([1, 16, 112, 112])
> features.1: torch.Size([1, 16, 56, 56])
> features.2: torch.Size([1, 24, 28, 28])
> features.3: torch.Size([1, 24, 28, 28])
> features.4: torch.Size([1, 40, 14, 14])
> features.5: torch.Size([1, 40, 14, 14])
> features.6: torch.Size([1, 40, 14, 14])
> features.7: torch.Size([1, 48, 14, 14])
> features.8: torch.Size([1, 48, 14, 14])
> features.9: torch.Size([1, 96, 7, 7])
> features.10: torch.Size([1, 96, 7, 7])
> features.11: torch.Size([1, 96, 7, 7])
> features.12: torch.Size([1, 576, 7, 7])


In [19]:

def pretty_print_studentv3(student_name):
    
    model = models.get_untrained_model(student_name, num_classes=7)
    nparams, n_trainable_params = utils.count_params_in_model(model)
    return model
    num_features = 4
    print(f"Student `{student_name}` (params: {nparams / 1e6:.1f}M)")
    
    x = torch.randn(1, 3, 224, 224)
    _, arr_outputs = utils.interceptor.forward_and_intercept_intermediate_layers(
        model, x,
        layers=[f"layer{ix+1}" for ix in range(num_features)],
        detach_output=False
    )

    for lix, output in enumerate(arr_outputs):
        print(f"> features.{lix}: {output.shape}")

pretty_print_studentv3(
"student-32-24-16-8",
)

StudentModel(
  (stem): Sequential(
    (0): ConvBN(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ReLU()
    (2): ConvBN(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ConvBN(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ReLU()
    (2): ConvBN(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_run

In [ ]:
raise

In [ ]:
models.get_untrained_model("student-mobilenets", num_classes=7)

In [ ]:
models.get_untrained_model("student-bmobilenets", num_classes=7)

In [ ]:
pretty_print_student("student-mobilenetxs")

In [ ]:
pretty_print_student("student-mobilenetxxs")

In [ ]:
raise

In [ ]:
def pretty_print2(model, arr_layers):    
    
    x = torch.randn(1, 3, 224, 224)
    _, arr_outputs = utils.interceptor.forward_and_intercept_intermediate_layers(
        model, x,
        layers=arr_layers,
        detach_output=False
    )
    
    mapping = dict()

    for lix, output in enumerate(arr_outputs):
        mapping[arr_layers[lix]] = output.shape
    return mapping

pretty_print2(models.get_trained_model("imagenet-resnet18-tv"), ["layer3", "layer4"])

In [ ]:
pretty_print2(models.get_trained_model("imagenet-resnet50-tv"), ["layer3", "layer4"])

In [ ]:
pretty_print2(models.get_trained_model("imagenet-vgg16-tv"), ["features.23", "features.30"])

In [ ]:
pretty_print2(models.get_trained_model("imagenet-nfnetf0-dm"), ["stages.2", "stages.3"])

In [ ]:
pretty_print2(models.get_trained_model("imagenet-mobilenetl-tv"), ["features.12", "features.16"])

In [ ]:
def ano(student):
    
    for teacher in ["imagenet-resnet18-tv", "imagenet-resnet50-tv", "imagenet-vgg16-tv", "imagenet-nfnetf0-dm"]:
        if "cifar" in teacher:
            continue
            x = torch.randn(1, 3, 32, 32)
        else:    
            x = torch.randn(1, 3, 224, 224)
        pairs = constants.DEFAULT_TEACHER_STUDENT_LAYER_MAPPING[teacher].split(",")
        arr_teacher_layers = list(map(lambda p: p.split(":")[0], pairs))
        arr_student_layers = list(map(lambda p: p.split(":")[1], pairs))

        _, arr_teacher_outputs = utils.interceptor.forward_and_intercept_intermediate_layers(
            models.get_trained_model(teacher), x,
            layers=arr_teacher_layers,
            detach_output=False
        )


        _, arr_student_outputs = utils.interceptor.forward_and_intercept_intermediate_layers(
            models.get_untrained_model(student, num_classes=10), x,
            layers=arr_student_layers,
            detach_output=False
        )
        
        print(f"Teacher `{teacher}` -> Student `{student}`")
        for pix in range(len(arr_teacher_layers)):
            teacher_shape = tuple(arr_teacher_outputs[pix].shape)
            student_shape = tuple(arr_student_outputs[pix].shape)
            assert teacher_shape[1] > student_shape[1], f"{teacher_shape} vs {student_shape}"
            print(f">>> {arr_teacher_layers[pix]}{teacher_shape} -> {arr_student_layers[pix]}{student_shape} ")
            
ano(student = "student-mobilenetxs")

In [ ]:
ano(student = "student-mobilenetxxs")

# ViT Teacher and Student

In [ ]:
def count_trainable_params(model):
    nparams, n_trainable_params = utils.count_params_in_model(model)
   
    print(f"params: {nparams / 1e6:.1f}M)")
    
count_trainable_params(models.get_untrained_model("vitstudent-48", num_classes=10))

In [ ]:
count_trainable_params(models.get_untrained_model("vitstudent-60", num_classes=10))

In [ ]:
count_trainable_params(models.get_untrained_model("vitstudent-72", num_classes=10))

In [ ]:
count_trainable_params(models.get_untrained_model("vitstudent-132", num_classes=10))

In [ ]:
def ano():
    model = models.get_trained_model("imagenet-vitb-tv")
    utils.modify_last_layer_for_subclasses(model, list(range(10)))
    count_trainable_params(model)
ano()

In [ ]:
len(models.get_trained_model("imagenet-vitb-tv").encoder.layers)